<a href="https://colab.research.google.com/github/ealmeida04/logica-programacao/blob/main/otimizacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importando bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.sql.functions import month, year, col
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [2]:
# Criando a sessão Spark
spark = SparkSession.builder.appName("YouTubeDataPreparation").getOrCreate()

**LENDO OS ARQUIVOS DE IMPORTAÇÃO**

In [3]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  uploaded_file_name = fn

print(f"The uploaded file name is: {uploaded_file_name}")

Saving videos-preparados.snappy.parquet to videos-preparados.snappy.parquet
Saving videos-comments-tratados.snappy.parquet to videos-comments-tratados.snappy.parquet
User uploaded file "videos-preparados.snappy.parquet" with length 245808 bytes
User uploaded file "videos-comments-tratados.snappy.parquet" with length 1994329 bytes
The uploaded file name is: videos-comments-tratados.snappy.parquet


In [6]:
df_video = spark.read.parquet('videos-preparados.snappy.parquet')
df_comments = spark.read.parquet('videos-comments-tratados.snappy.parquet')

**Crie tabelas temporárias para ambos os dataframe**

In [7]:
df_video.createOrReplaceTempView('df_video_temp')
df_comments.createOrReplaceTempView('df_comments_temp')

**Faça um join das tabelas criadas anteriormente utilizando o spark.sql no dataframe 'join_video_comments'**

In [9]:
join_video_comments = spark.sql("""
    SELECT
        v.*,
        c.Comment,
        c.Sentiment,
        c.`Likes Comment`
    FROM
        df_video_temp v
    JOIN
        df_comments_temp c
    ON
        v.`Video ID` = c.`Video ID`
""")

**Faça as mesmas etapas anteriores (1,2,3,4) utilizando repartição e coalesce**

In [21]:
# Verificando o número atual de partições do DataFrame 'join_video_comments'
print(f"Número de partições de join_video_comments antes: {join_video_comments.rdd.getNumPartitions()}")

Número de partições de join_video_comments antes: 1


In [20]:
# Demonstração de repartition(): para 20 partições

repartitioned_df = join_video_comments.repartition(20)
print(f"Número de partições de join_video_comments após repartition(20): {repartitioned_df.rdd.getNumPartitions()}")

Número de partições de join_video_comments após repartition(20): 20


In [22]:
# Demonstração de coalesce(): Vamos reduzir para 5 partições
# DataFrame com mais partições para a demonstração de coalesce
# Vou usar o DataFrame original e o reparticionar para um número maior temporariamente

df_for_coalesce = join_video_comments.repartition(15) # Aumenta para 15 partições para demonstrar coalesce
print(f"Número de partições do DataFrame temporário antes de coalesce: {df_for_coalesce.rdd.getNumPartitions()}")

coalesced_df = df_for_coalesce.coalesce(5)
print(f"Número de partições após coalesce(5) no DataFrame temporário: {coalesced_df.rdd.getNumPartitions()}")

# Observação: `repartition` pode aumentar ou diminuir, sempre realizando um shuffle.
# `coalesce` só diminui, tentando evitar um shuffle completo se possível.

Número de partições do DataFrame temporário antes de coalesce: 15
Número de partições após coalesce(5) no DataFrame temporário: 5


**Utilize o explicado para entender melhor as duas formas de realizar as etapas e refaça novamente as etapas anteriores (1,2,3,4), utilizando tudo o que você já aprendeu para realizar o join e filtrar apenas com os dados necessários**

## Recarregar DataFrames com Cache

Recarregar os arquivos Parquet em novos DataFrames (df_video_optimized e df_comments_optimized) e aplicar .cache() para otimização.


In [23]:
df_video_optimized = spark.read.parquet('videos-preparados.snappy.parquet')
df_video_optimized.cache()
print("df_video_optimized criado e cacheado.")

df_comments_optimized = spark.read.parquet('videos-comments-tratados.snappy.parquet')
df_comments_optimized.cache()
print("df_comments_optimized criado e cacheado.")

df_video_optimized criado e cacheado.
df_comments_optimized criado e cacheado.


O próximo passo é criar visualizações temporárias otimizadas a partir dos DataFrames em cache, conforme especificado na tarefa principal, para permitir consultas Spark SQL.



In [24]:
df_video_optimized.createOrReplaceTempView('df_video_optimized_temp')
print("Temporary view 'df_video_optimized_temp' criada.")

df_comments_optimized.createOrReplaceTempView('df_comments_optimized_temp')
print("Temporary view 'df_comments_optimized_temp' criada.")

Temporary view 'df_video_optimized_temp' criada.
Temporary view 'df_comments_optimized_temp' criada.


O próximo passo é realizar uma junção Spark SQL usando as visualizações temporárias otimizadas, selecionando as colunas especificadas de df_video_optimized_temp e df_comments_optimized_temp com base no 'ID do Vídeo', conforme descrito na tarefa principal.


In [25]:
join_video_comments_optimized = spark.sql("""
    SELECT
        v.`Title`,
        v.`Video ID`,
        v.`Published At`,
        v.`Keyword`,
        v.`Likes`,
        v.`Comments`,
        v.`Views`,
        v.`Interaction`,
        v.`Year`,
        v.`Month`,
        v.`Keyword Index`,
        v.`Features PCA`,
        v.`Features Normal`,
        v.`Features`,
        c.`Comment`,
        c.`Sentiment`,
        c.`Likes Comment`
    FROM
        df_video_optimized_temp v
    JOIN
        df_comments_optimized_temp c
    ON
        v.`Video ID` = c.`Video ID`
""")
print("Optimized join DataFrame 'join_video_comments_optimized' created.")

Optimized join DataFrame 'join_video_comments_optimized' created.


reparticionar o DataFrame `join_video_comments_optimized` em 10 partições e, em seguida, exibir o número de partições e seu esquema, conforme exigido pela tarefa principal.


In [27]:
join_video_comments_repartitioned = join_video_comments_optimized.repartition(10)
print(f"Number of partitions after repartitioning: {join_video_comments_repartitioned.rdd.getNumPartitions()}")
print("Schema of the repartitioned DataFrame:")
join_video_comments_repartitioned.printSchema()

Number of partitions after repartitioning: 10
Schema of the repartitioned DataFrame:
root
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Keyword Index: double (nullable = true)
 |-- Features PCA: vector (nullable = true)
 |-- Features Normal: vector (nullable = true)
 |-- Features: vector (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Sentiment: integer (nullable = true)
 |-- Likes Comment: integer (nullable = true)



## Resumo:

### Principais Descobertas da Análise de Dados

* Dois arquivos Parquet, `videos-preparados.snappy.parquet` e `videos-comments-tratados.snappy.parquet`, foram carregados com sucesso em DataFrames do Spark (`df_video_optimized` e `df_comments_optimized`) e armazenados em cache para otimizar o desempenho.

* Visualizações temporárias otimizadas (`df_video_optimized_temp` e `df_comments_optimized_temp`) foram criadas com sucesso a partir dos DataFrames em cache para facilitar as consultas Spark SQL.

* Uma consulta Spark SQL realizou com sucesso uma junção interna entre os dados de vídeo e comentários na coluna 'ID do Vídeo', selecionando 17 colunas específicas para criar o DataFrame `join_video_comments_optimized`.

* O DataFrame resultante da junção foi explicitamente reparticionado em 10 partições, e essa contagem de partições foi confirmada como sendo 10.
* O esquema do DataFrame reparticionado foi exibido, confirmando a presença de 17 colunas com tipos de dados incluindo `string`, `date`, `integer`, `double` e `vector`.

Insights - realizar análises de dados adicionais, engenharia de recursos ou treinamento de modelos no DataFrame `join_video_comments_repartitioned`

In [28]:
print("### 1. Schema of join_video_comments_repartitioned:")
join_video_comments_repartitioned.printSchema()

print("\n### 2. First 5 rows of join_video_comments_repartitioned:")
join_video_comments_repartitioned.show(5)

print("\n### 3. Descriptive statistics for join_video_comments_repartitioned:")
join_video_comments_repartitioned.describe().show()

### 1. Schema of join_video_comments_repartitioned:
root
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Keyword Index: double (nullable = true)
 |-- Features PCA: vector (nullable = true)
 |-- Features Normal: vector (nullable = true)
 |-- Features: vector (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Sentiment: integer (nullable = true)
 |-- Likes Comment: integer (nullable = true)


### 2. First 5 rows of join_video_comments_repartitioned:
+--------------------+-----------+------------+----------------+-------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+-----------------

In [29]:
print("\n### 4. Distinct counts for categorical columns:")
print(f"Distinct count for 'Keyword': {join_video_comments_repartitioned.select('Keyword').distinct().count()}")
print(f"Distinct count for 'Sentiment': {join_video_comments_repartitioned.select('Sentiment').distinct().count()}")


### 4. Distinct counts for categorical columns:
Distinct count for 'Keyword': 41
Distinct count for 'Sentiment': 154


In [30]:
print("\n### 5. Count of null values per column:")
for col_name in join_video_comments_repartitioned.columns:
    null_count = join_video_comments_repartitioned.filter(col(col_name).isNull()).count()
    print(f"Column '{col_name}': {null_count} null values")


### 5. Count of null values per column:
Column 'Title': 0 null values
Column 'Video ID': 0 null values
Column 'Published At': 0 null values
Column 'Keyword': 0 null values
Column 'Likes': 0 null values
Column 'Comments': 0 null values
Column 'Views': 0 null values
Column 'Interaction': 0 null values
Column 'Year': 0 null values
Column 'Month': 0 null values
Column 'Keyword Index': 0 null values
Column 'Features PCA': 0 null values
Column 'Features Normal': 0 null values
Column 'Features': 0 null values
Column 'Comment': 1 null values
Column 'Sentiment': 3135 null values
Column 'Likes Comment': 3338 null values


## Engenharia de Recursos

Criar novas features a partir das colunas existentes no DataFrame `join_video_comments_repartitioned` para melhorar o desempenho de modelos preditivos, e tratar valores inconsistentes e ausentes.


In [31]:
from pyspark.sql.functions import when, col

# 1. Substituir valores negativos nas colunas 'Likes', 'Comments', 'Views' e 'Interaction' por 0
join_video_comments_cleaned = join_video_comments_repartitioned.withColumn(
    "Likes", when(col("Likes") < 0, 0).otherwise(col("Likes"))
).withColumn(
    "Comments", when(col("Comments") < 0, 0).otherwise(col("Comments"))
).withColumn(
    "Views", when(col("Views") < 0, 0).otherwise(col("Views"))
).withColumn(
    "Interaction", when(col("Interaction") < 0, 0).otherwise(col("Interaction"))
)

print("Negative values replaced with 0 in 'Likes', 'Comments', 'Views', and 'Interaction'.")
# Display some rows to verify the changes if any negative values were present
# join_video_comments_cleaned.select('Likes', 'Comments', 'Views', 'Interaction').filter((col('Likes') < 0) | (col('Comments') < 0) | (col('Views') < 0) | (col('Interaction') < 0)).show()


Negative values replaced with 0 in 'Likes', 'Comments', 'Views', and 'Interaction'.


In [32]:
from pyspark.sql.functions import median

# 2. Impute missing values in 'Sentiment' and 'Likes Comment' with their medians
# Calculate median for 'Sentiment'
median_sentiment = join_video_comments_cleaned.agg(median("Sentiment")).collect()[0][0]
# Calculate median for 'Likes Comment'
median_likes_comment = join_video_comments_cleaned.agg(median("Likes Comment")).collect()[0][0]

# Impute missing values
join_video_comments_imputed = join_video_comments_cleaned.na.fill({"Sentiment": median_sentiment, "Likes Comment": median_likes_comment})

print(f"Missing values in 'Sentiment' imputed with median: {median_sentiment}")
print(f"Missing values in 'Likes Comment' imputed with median: {median_likes_comment}")

Missing values in 'Sentiment' imputed with median: 2.0
Missing values in 'Likes Comment' imputed with median: 26.0


In [33]:
join_video_comments_comment_filled = join_video_comments_imputed.na.fill({'Comment': ''})

print("Missing values in 'Comment' replaced with an empty string.")

Missing values in 'Comment' replaced with an empty string.


In [34]:
from pyspark.sql.functions import dayofweek, dayofmonth, dayofyear

# 4. Criar novas features temporais a partir da coluna 'Published At'
join_video_comments_temporal_features = join_video_comments_comment_filled.withColumn(
    "day_of_week", dayofweek(col("Published At"))
).withColumn(
    "day_of_month", dayofmonth(col("Published At"))
).withColumn(
    "day_of_year", dayofyear(col("Published At"))
)

print("New temporal features 'day_of_week', 'day_of_month', and 'day_of_year' created.")

New temporal features 'day_of_week', 'day_of_month', and 'day_of_year' created.


In [36]:
from pyspark.sql.functions import lit

# 5. Criar novas features de interação: 'Likes_per_View' e 'Comments_per_View'
join_video_comments_final = join_video_comments_temporal_features.withColumn(
    "Likes_per_View",
    when(col("Views") == 0, lit(0)).otherwise(col("Likes") / col("Views"))
).withColumn(
    "Comments_per_View",
    when(col("Views") == 0, lit(0)).otherwise(col("Comments") / col("Views"))
)

print("New interaction features 'Likes_per_View' and 'Comments_per_View' created.")

New interaction features 'Likes_per_View' and 'Comments_per_View' created.


**Salve o seu join otimizado como 'join-videos-comments-otimizado' no formato parquet**

In [37]:
join_video_comments_optimized.write.mode("overwrite").parquet('join-videos-comments-otimizado.parquet')
print("DataFrame 'join_video_comments_optimized' salvo como 'join-videos-comments-otimizado.parquet'.")

DataFrame 'join_video_comments_optimized' salvo como 'join-videos-comments-otimizado.parquet'.


## Treinamento de Modelo

### Subtask:
Configurar e treinar um modelo de Machine Learning (por exemplo, um modelo de regressão para prever visualizações ou curtidas) usando as features do DataFrame `join_video_comments_final`, incluindo as features engenheiradas, e avaliar seu desempenho.


## Resumo:

### Principais Descobertas da Análise de Dados

* O conjunto de dados inicial continha 18.409 registros e 17 colunas, incluindo `Título`, `ID do Vídeo`, `Publicado em`, `Palavra-chave`, `Curtidas`, `Comentários`, `Visualizações`, `Interação`, `Comentário`, `Sentimento` e `Comentário de Curtidas`.

* Foram identificados problemas significativos de qualidade de dados:
* Valores negativos estavam presentes nas colunas `Curtidas`, `Comentários`, `Visualizações` e `Interação`, o que é uma anomalia para essas métricas (por exemplo, o valor mínimo de `Interação` foi -2130305273).

* A coluna `Visualizações` apresentou um valor máximo de 2147483647, sugerindo um possível problema de limite de dados ou estouro de inteiro.

* A coluna `Sentimento` apresentou uma amplitude incomum (de 0 a 19518) e uma alta cardinalidade (154 valores distintos), desviando-se da pontuação de sentimento típica.
* Valores ausentes foram observados em colunas-chave:
* `Comentário`: 1 valor nulo.
* `Sentimento`: 3135 valores nulos.
* `Curtidas Comentário`: 3338 valores nulos.
* **Etapas de Engenharia de Recursos e Limpeza de Dados:**

* Valores negativos em `Curtidas`, `Comentários`, `Visualizações` e `Interação` foram substituídos por 0.
* Valores numéricos ausentes em `Sentimento` foram imputados com sua mediana (2,0) e em `Curtidas Comentário` com sua mediana (26,0).

* O único valor ausente em `Comentário` foi substituído por uma string vazia.

* Novos recursos temporais (`dia_da_semana`, `dia_do_mês`, `dia_do_ano`) foram extraídos de `Publicado em`.

* Foram criadas variáveis ​​de interação (`Curtidas_por_visualização`, `Comentários_por_visualização`), com tratamento robusto para zero visualizações (`Views`) para evitar erros de divisão.